In [1]:
from ngo.ml.basisfunctions import *
from ngo.ml.quadrature import *

In [2]:
N_samples = 10
N = 200
n_t = 2
n_x = 10
n_y = 10
Q_t = 3
Q_x = 99
Q_y = 99

n_elements_t = 1
n_elements_x = 3
n_elements_y = 3

basis_t = BSplineBasis1D(h=n_t, p=1, C=0)
basis_x = BSplineBasis1D(h=n_x, p=3, C=2)
basis_y = BSplineBasis1D(h=n_y, p=3, C=2)

quadrature_t = GaussLegendreQuadrature(Q=[Q_t], n_elements=[n_elements_t])
quadrature_x = GaussLegendreQuadrature(Q=[Q_x], n_elements=[n_elements_x])
quadrature_y = GaussLegendreQuadrature(Q=[Q_y], n_elements=[n_elements_y])
t = quadrature_t.xi.reshape(quadrature_t.xi.shape[0],)
x = quadrature_x.xi.reshape(quadrature_x.xi.shape[0],) 
y = quadrature_y.xi.reshape(quadrature_y.xi.shape[0],)

X = t,x,y

quadrature = GaussLegendreQuadrature(Q=(Q_t,Q_x,Q_y), n_elements=(n_elements_t,n_elements_x,n_elements_y))
X_q = quadrature.xi

u_m = np.random.uniform(size=(N_samples,N))
U = u_m.reshape((N_samples,n_t, n_x, n_y))


In [ ]:
class TensorProductBasis:
    """
    Tensor-product (Kronecker) basis built from multiple 1D basis objects.

    This class combines several 1D basis objects into a d-dimensional basis by
    taking tensor products of the 1D basis functions. It evaluates a field
    given modal coefficients `u_m` at tensor-product evaluation points and
    computes partial derivatives by replacing a 1D basis with its derivative
    in the corresponding dimension.

    Requirements for each 1D basis object:
      - .h attribute: number of basis functions in that 1D basis
      - .forward(xi): returns shape (n_basis_i, n_points_i)
      - .grad(xi): returns shape (n_basis_i, n_points_i)

    Attributes:
        bases (tuple): Sequence of 1D basis objects.
        d (int): Number of dimensions (len(bases)).
        h (tuple): Number of basis functions per dimension (h1, ..., hd).
    """
    def __init__(self, bases):
        """
        Initialize the tensorized basis.

        Args:
            bases (list): A list of 1D basis objects.
        """
        self.bases = bases
        self.d = len(bases)
        self.h = ()
        for i in range(len(bases)):
            self.h += (bases[i].h,)

    def forward(self, u_m, x):
        """
        Evaluate the tensor-product expansion at provided points.

        Args:
            u_m (np.ndarray): Modal coefficients with shape (N_samples, prod(h)) or
                              (N_samples, h1, h2, ..., hd). N_samples is batch size.
            x (tuple/list): Sequence of point arrays for each dimension.
                            x[i] has shape (n_points_i,).

        Returns:
            np.ndarray: Evaluated values at tensor-product grid points,
                        shape (N, n_points_1, ..., n_points_d).
        """
        # reshape coefficients into (N_samples, h1, h2, ..., hd)
        u_q = u_m.reshape(u_m.shape[0],*self.h)
        # sequentially contract coefficient tensor with each 1D basis evaluation
        for i in range(self.d):
           # compute basis evaluation matrix of dimension i
           Phi_i = self.bases[i].forward(x[i])
            # tensordot contracts the basis index (axis 1 of u_q) with axis 0 of Phi_i
           u_q = np.tensordot(u_q, Phi_i, axes=(1,0))
        # final shape is (N_samples, n_points_1, ..., n_points_d)
        return u_q


    def grad(self, u_m, x):
        """
        Compute partial derivatives of the tensor-product expansion.
        For each spatial dimension i the method replaces the 1D basis eval in
        that dimension with its derivative and contracts as in `forward`.

        Args:
            u_m (np.ndarray): Modal coefficients, shape (N, prod(h)) or (N, h1, ..., hd).
            x (tuple/list): Sequence of point arrays for each dimension.
                            x[i] has shape (n_points_i,).
        Returns:
            np.ndarray: Gradients at tensor-product points with shape
                        (N, n_points_1, ..., n_points_d, d). The last axis indexes
                        partial derivatives w.r.t. each coordinate.
        """
        #Compute function evaluation and gradient matrices
        Phi_i = tuple(b.forward(xi) for b, xi in zip(self.bases, x))
        dPhi_i = tuple(b.grad(xi) for b, xi in zip(self.bases, x))
        
        # Compute derivative for each dimension by contracting with dPhi in that dim.
        gradu_q = ()
        for i in range(self.d):
            # start from coefficients reshaped to (N, h1, ..., hd)
            u_q = u_m.reshape(u_m.shape[0],*self.h)
            for j in range(self.d):
                if j==i:
                    # use derivative matrix in the i-th dimension
                    u_q = np.tensordot(u_q, dPhi_i[j], axes=(1,0))
                else:
                    # use value matrix in other dimensions
                    u_q = np.tensordot(u_q, Phi_i[j], axes=(1,0))
            # append gradient in direction i to gradu_q tuple
            gradu_q += (u_q,)
        #Reshape such that the last axis indexes the coordinate to which derivative has been taken
        gradu_q = np.stack(gradu_q, axis=-1)
        return gradu_q


In [4]:
basis1 = TensorProductBasis(bases=(basis_t,basis_x,basis_y))

In [5]:
result1 = basis1.forward(u_m, X)

In [6]:
result1.shape

(10, 3, 99, 99)

In [7]:
basis2 = TensorizedBasis(bases=[basis_t,basis_x,basis_y])

In [8]:
Phi2 = basis2.forward(X_q)

ValueError: Size of label 'N' for operand 1 (2) does not match previous terms (10).

In [ ]:
result2 = np.einsum('Nn,qn->Nq', u_m, Phi2).reshape(N_samples,Q_t,Q_x,Q_y)

In [ ]:
result2 - result1

array([[[[-2.22044605e-16,  0.00000000e+00, -2.22044605e-16, ...,
          -1.11022302e-16,  0.00000000e+00,  0.00000000e+00],
         [-1.11022302e-16,  0.00000000e+00,  1.11022302e-16, ...,
          -2.22044605e-16, -1.11022302e-16, -2.22044605e-16],
         [-2.22044605e-16, -1.11022302e-16,  0.00000000e+00, ...,
          -2.22044605e-16,  0.00000000e+00, -1.11022302e-16],
         ...,
         [ 3.33066907e-16, -3.33066907e-16,  0.00000000e+00, ...,
           2.77555756e-17,  8.32667268e-17,  2.77555756e-17],
         [ 1.11022302e-16,  0.00000000e+00, -1.11022302e-16, ...,
           2.77555756e-17,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          -2.77555756e-17,  2.77555756e-17,  0.00000000e+00]],

        [[-2.22044605e-16,  3.33066907e-16,  2.22044605e-16, ...,
           1.11022302e-16,  0.00000000e+00,  3.33066907e-16],
         [ 3.33066907e-16,  0.00000000e+00, -2.22044605e-16, ...,
           0.00000000e